# Experiment Type Recovery Analysis

This notebook recomputes degradation and recovery metrics directly from the raw `baseline → chaos → recovery` measurements recorded in each experiment log. It then aggregates the results per experiment type, contrasts them with the success flags stored in the logs, and visualises overall recovery effectiveness alongside agent effort.

In [ ]:
import json
from pathlib import Path
from typing import Dict, Any, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams.update({
    'figure.figsize': (14, 7),
    'axes.titlesize': 16,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
})

## 1. Select and Load Experiment Logs

Update `DATA_DIRECTORIES` to point to the batches you want to analyse. All JSON experiment logs inside the listed directories will be processed.

In [ ]:
DATA_DIRECTORIES = [
    # Path('DATASET_LOGS/gpt-5/batch_20251112_104955_app-smart-city'),
    # Path('DATASET_LOGS/gpt-5/batch_20251113_090217_app-smart-manufacturing'),
    # Path('DATASET_LOGS/gpt-5/batch_20251113_131452_app-iot-gateway'),
    Path('DATASET_LOGS/gpt-5-mini/batch_20251114_100655_app-smart-city')
]

available_files = []
for directory in DATA_DIRECTORIES:
    if not directory.exists():
        print(f'⚠️ Skipping missing directory: {directory}')
        continue
    matched = sorted(directory.glob('*.json'))
    available_files.extend(matched)
    print(f'✓ {directory} → {len(matched)} files')

if not available_files:
    raise RuntimeError('No JSON log files found. Update DATA_DIRECTORIES to valid batches.')

print(f"\nTotal: {len(available_files)} experiment logs across {len({fp.parent for fp in available_files})} directory(ies).")

## 2. Metric Extraction Helpers

The helper functions below classify experiments and derive degradation and recovery metrics from the baseline, chaos, and recovery phases.

In [ ]:
def classify_experiment(fault_type: Optional[str], target_mode: Optional[str]) -> str:
    '''Map raw fault type and target mode to high-level experiment categories.'''
    fault_type = (fault_type or 'unknown').lower()
    target_mode = (target_mode or '').lower()

    if fault_type == 'baseline':
        return 'Baseline'
    if fault_type == 'pod_failure':
        return 'Pod Failure (All Pods)' if target_mode == 'all_pods_app' else 'Pod Failure (One Pod)'
    if fault_type == 'cpu_stress':
        return 'Pod CPU Stress (All Pods)' if target_mode == 'all_pods_app' else 'Pod CPU Stress (One Pod)'
    if fault_type == 'memory_stress':
        return 'Pod Memory Stress (All Pods)' if target_mode == 'all_pods_app' else 'Pod Memory Stress (One Pod)'
    if fault_type == 'node_cpu_stress':
        return 'Node CPU Stress'
    if fault_type == 'node_mem_stress':
        return 'Node Memory Stress'
    return f'Other: {fault_type}'.title()


def compute_recovery_index(baseline: float, chaos: float, recovery: float, *, higher_is_better: bool) -> float:
    '''Return the recovery index (percentage of degradation regained).'''
    eps = 1e-9

    if higher_is_better:
        delta = baseline - chaos
        if abs(delta) < eps:
            return 100.0 if recovery >= baseline else 0.0
        progress = recovery - chaos
    else:
        delta = chaos - baseline
        if abs(delta) < eps:
            return 100.0 if recovery <= baseline else 0.0
        progress = chaos - recovery

    index = (progress / delta) * 100
    return float(np.clip(index, 0.0, 120.0))


def process_log(filepath: Path) -> Dict[str, Any]:
    with filepath.open() as f:
        data = json.load(f)

    baseline = data.get('baseline_metrics', {}) or {}
    chaos = data.get('chaos_metrics', {}) or {}
    recovery = data.get('recovery_metrics', {}) or {}
    recovery_eval = data.get('recovery_evaluation', {}) or {}
    location_eval = data.get('location_evaluation', {}) or {}

    baseline_lat = baseline.get('p95_latency_seconds') or 0.0
    baseline_sr = baseline.get('success_rate_percent') or 0.0
    chaos_lat = chaos.get('p95_latency_seconds', baseline_lat)
    chaos_sr = chaos.get('success_rate_percent', baseline_sr)
    recovery_lat = recovery.get('p95_latency_seconds', chaos_lat)
    recovery_sr = recovery.get('success_rate_percent', chaos_sr)

    latency_deg_pct = ((chaos_lat - baseline_lat) / baseline_lat * 100) if baseline_lat > 0 else 0.0
    success_deg_pct = ((baseline_sr - chaos_sr) / baseline_sr * 100) if baseline_sr > 0 else 0.0

    latency_recovery_pct = compute_recovery_index(baseline_lat, chaos_lat, recovery_lat, higher_is_better=False)
    success_recovery_pct = compute_recovery_index(baseline_sr, chaos_sr, recovery_sr, higher_is_better=True)

    latency_within_20 = recovery_lat <= baseline_lat * 1.20 if baseline_lat > 0 else True
    success_within_1 = abs(recovery_sr - baseline_sr) <= 1.0
    recovery_success = bool(latency_within_20 and success_within_1)

    chaos_config = data.get('chaos_config', {}) or {}
    target_cfg = chaos_config.get('target') or {}

    # Extract app type from filepath (e.g., "app-smart-city" from "batch_20251112_104955_app-smart-city")
    app_type = 'unknown'
    path_parts = filepath.parts
    for part in path_parts:
        if part.startswith('app-'):
            app_type = part
            break
    # Fallback: try to extract from parent directory name
    if app_type == 'unknown' and filepath.parent:
        parent_name = filepath.parent.name
        if 'app-' in parent_name:
            # Extract app-* part from parent name
            import re
            match = re.search(r'app-[a-z-]+', parent_name)
            if match:
                app_type = match.group(0)

    # Calculate total sleep time from agent_trace
    agent_trace = data.get('agent_trace', [])
    total_sleep_seconds = 0.0
    for step in agent_trace:
        if step.get('action') == 'sleep_seconds':
            # Extract duration from action_args or observation
            action_args = step.get('action_args', {})
            observation = step.get('observation', {})
            duration = action_args.get('duration') or observation.get('duration') or 0.0
            total_sleep_seconds += float(duration)

    return {
        'file': filepath.name,
        'app_type': app_type,
        'experiment_id': data.get('experiment_id'),
        'category': classify_experiment(
            chaos_config.get('fault_type', chaos_config.get('type')),
            target_cfg.get('mode')
        ),
        'baseline_latency': baseline_lat,
        'chaos_latency': chaos_lat,
        'recovery_latency': recovery_lat,
        'baseline_success_rate': baseline_sr,
        'chaos_success_rate': chaos_sr,
        'recovery_success_rate': recovery_sr,
        'latency_degradation_pct': latency_deg_pct,
        'success_degradation_pct': success_deg_pct,
        'latency_recovery_pct': latency_recovery_pct,
        'success_recovery_pct': success_recovery_pct,
        'within20_logged': bool(recovery_eval.get('within_20pct_baseline', False)),
        'within20_recalc': recovery_success,
        'latency_within_20pct': latency_within_20,
        'success_within_1pct': success_within_1,
        'recovery_success': recovery_success,
        'recovery_index_latency': latency_recovery_pct,
        'recovery_index_success': success_recovery_pct,
        'recovery_index_composite': (latency_recovery_pct + success_recovery_pct) / 2,
        'location_identified': bool(location_eval.get('location_identified', False)),
        'agent_duration_seconds': data.get('agent_duration_seconds', 0.0),
        'agent_duration_minutes': (data.get('agent_duration_seconds', 0.0) / 60.0),
        'agent_steps': len(agent_trace),
        'agent_sleep_seconds': total_sleep_seconds,
        'agent_sleep_minutes': total_sleep_seconds / 60.0,
    }

records = [process_log(fp) for fp in available_files]
df = pd.DataFrame(records)
df.sort_values('category', inplace=True)

print(f"Processed {len(df)} experiments across {df['category'].nunique()} experiment types.")
df.head()

## 3. Degradation and Recovery Overview

Average chaos degradation relative to baseline (positive ⇒ worse) and the percentage of that degradation that the system recovered.

In [ ]:
degradation_summary = (
    df.groupby('category')[
        ['latency_degradation_pct', 'success_degradation_pct', 'latency_recovery_pct', 'success_recovery_pct']
    ]
    .mean()
    .rename(columns={
        'latency_degradation_pct': 'Latency Δ during chaos (%)',
        'success_degradation_pct': 'Success-rate Δ during chaos (%)',
        'latency_recovery_pct': 'Latency recovery (%)',
        'success_recovery_pct': 'Success-rate recovery (%)',
    })
    .round(2)
)

degradation_summary

## 4. Baseline → Chaos → Recovery Levels

Average p95 latency (left) and success rate (right) across experiment types.

In [ ]:
import matplotlib.ticker as mtick

latency_levels = (
    df.groupby('category')[['baseline_latency', 'chaos_latency', 'recovery_latency']]
      .mean()
      .reset_index()
      .melt(id_vars='category', var_name='phase', value_name='latency_seconds')
)

success_levels = (
    df.groupby('category')[['baseline_success_rate', 'chaos_success_rate', 'recovery_success_rate']]
      .mean()
      .reset_index()
      .melt(id_vars='category', var_name='phase', value_name='success_rate')
)


# Create a "failure rate" column for visualization
failure_levels = success_levels.copy()
failure_levels['failure_rate'] = 100 - failure_levels['success_rate']

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# --- Latency plot ---
sns.barplot(
    data=latency_levels,
    x='category', y='latency_seconds', hue='phase',
    hue_order=['baseline_latency', 'chaos_latency', 'recovery_latency'],
    palette=['#43a047', '#e53935', '#1e88e5'],
    ax=axes[0],
    dodge=True
)
axes[0].set_title('Average p95 Latency by Phase')
axes[0].set_ylabel('Latency (s)')
axes[0].legend_.remove()  # remove legend

# --- Success degradation plot (log scale) ---
sns.barplot(
    data=failure_levels,
    x='category', y='failure_rate', hue='phase',
    hue_order=['baseline_success_rate', 'chaos_success_rate', 'recovery_success_rate'],
    palette=['#43a047', '#e53935', '#1e88e5'],
    ax=axes[1],
    dodge=True
)
axes[1].set_title('Average Success Rate Degradation by Phase')
axes[1].set_ylabel('Failure (%)')
axes[1].set_xlabel('Experiment Type')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
axes[1].legend_.remove()  # remove legend

# --- Single shared legend ---
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#43a047', label='Baseline'),
    Patch(facecolor='#e53935', label='Chaos'),
    Patch(facecolor='#1e88e5', label='Recovery')
]
axes[1].legend(handles=legend_elements, title='Phase', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()


## 5. Recovery vs. Location Identification

Compare how often the system recovered (latency ≤ 20% & success rate ≤ 1%) with how often the agent identified the correct problem location.

In [ ]:
recovery_vs_location = (
    df.groupby('category')[['recovery_success', 'location_identified']]
      .mean()
      .rename(columns={'recovery_success': 'Recovery success', 'location_identified': 'Location identified'})
      .reset_index()
)

fig, ax = plt.subplots(figsize=(14, 6))
width = 0.35
positions = np.arange(len(recovery_vs_location))

ax.bar(positions - width/2, recovery_vs_location['Recovery success'] * 100, width, label='Recovery success', color='#1e88e5')
ax.bar(positions + width/2, recovery_vs_location['Location identified'] * 100, width, label='Location identified', color='#fdd835')

ax.set_xticks(positions)
ax.set_xticklabels(recovery_vs_location['category'], rotation=45, ha='right')
ax.set_ylabel('Rate (%)')
ax.set_title('Average Recovery vs. Location Identification')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 7. Agent Effort: Execution Time and Rounds

Average agent runtime and number of steps per experiment type.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- Compute metrics ---
df['agent_active_minutes'] = (df['agent_duration_minutes'] - df['agent_sleep_minutes']).clip(lower=0)

# Sort categories
categories_order = df.groupby('category')['agent_duration_minutes'].mean().sort_values().index
df['category'] = pd.Categorical(df['category'], categories=categories_order, ordered=True)

# Summary
summary = df.groupby('category').agg({
    'agent_duration_minutes': 'mean',
    'agent_sleep_minutes': 'mean',
}).reset_index()

# --- Plot setup ---
sns.set_theme(style="white", font_scale=1.2)

fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(summary))
width = 0.65   # increased bar width

# Colors (Purple + Green)
total_color = "#6A1B9A"   # deep purple
sleep_color = "#2E7D32"   # strong natural green

# Total duration bars
ax.bar(
    x,
    summary['agent_duration_minutes'],
    color=total_color,
    alpha=0.55,
    width=width,
    label="Total Duration"
)

# Sleep time bars (same width)
ax.bar(
    x,
    summary['agent_sleep_minutes'],
    color=sleep_color,
    alpha=0.85,
    width=width,
    label="Sleep Time"
)

# Annotations (all black text)
for i, row in summary.iterrows():
    ax.text(
        i,
        row['agent_duration_minutes'] + 0.3,
        f"{row['agent_duration_minutes']:.1f} min",
        ha='center',
        va='bottom',
        fontsize=10,
        fontweight='bold',
        color='black'
    )
    ax.text(
        i,
        row['agent_sleep_minutes'] + 0.3,
        f"{row['agent_sleep_minutes']:.1f} min",
        ha='center',
        va='bottom',
        fontsize=10,
        fontweight='bold',
        color='black'
    )

# --- Styling ---
ax.set_xticks(x)
ax.set_xticklabels(summary['category'], rotation=30, ha='right')
ax.set_ylabel("Duration (minutes)")

ax.set_title(
    "Average Agent Duration and Sleep Time per Experiment Category",
    fontsize=14,
    fontweight='bold'
)

# Horizontal lines only (soft & clean)
ax.yaxis.grid(True, linestyle='--', alpha=0.25)
ax.xaxis.grid(False)

# Cap Y axis slightly above max value
ymax = summary['agent_duration_minutes'].max()
ax.set_ylim(0, ymax * 1.15)

ax.legend(frameon=False, loc='upper left')

plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# Revert to the 'cool' color palette
palette = sns.color_palette("cool", n_colors=len(categories_order))


sns.set_theme(style="whitegrid", font_scale=1.2)
fig, ax = plt.subplots(figsize=(10, 6))

# Revert to the 'cool' color palette
# Assuming categories_order is defined and available
# palette = sns.color_palette("cool", n_colors=len(categories_order)) 
# Using a placeholder for categories_order and df for the code presentation

# --- Begin Violin Plot Changes ---

# Use sns.violinplot
# inner='quartile' shows the median (center horizontal line) and the first/third quartiles
# inner='quartile' is a good substitute for showing the median as requested
sns.violinplot(
    data=df,
    x='category',
    y='agent_steps',
    hue='category',
    palette=palette, # Assuming 'palette' is defined as in the original code
    order=categories_order, # Assuming 'categories_order' is defined
    legend=False,
    inner='quartile', # <-- Shows the quartiles and the median line
    # cut=0,
    # scale='width',
    ax=ax
)

# --- End Violin Plot Changes ---


# Ticks and labels (kept the same)
ax.set_xticks(np.arange(len(categories_order)))
ax.set_xticklabels([str(c) for c in categories_order], rotation=45, ha='right')
ax.set_title("Agent Steps Distribution per Experiment (Violin Plot with Quartiles)", fontsize=14, fontweight='bold')
ax.set_xlabel("")
ax.set_ylabel("Steps / Rounds")
ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

## 8. Threshold Compliance per Experiment

Each point shows whether an experiment satisfied the latency ≤ 20% and success rate ≤ 1% thresholds.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- Prepare long-format compliance data ---
compliance_long = (
    df[['experiment_id', 'category', 'latency_within_20pct', 'success_within_1pct', 'location_identified']]
      .melt(id_vars=['experiment_id', 'category'], var_name='metric', value_name='within_threshold')
)

# Map readable labels
label_map = {
    'latency_within_20pct': 'Latency ≤ 20% baseline',
    'success_within_1pct': 'Success rate ≤ 1% diff',
    'location_identified': 'Location identified'
}
compliance_long['metric'] = compliance_long['metric'].map(label_map)

# Shorten experiment IDs for plotting
def shorten_experiment_id(exp_id):
    if pd.isna(exp_id): 
        return exp_id
    parts = exp_id.split('-')
    if len(parts) > 4:
        return '-'.join(parts[:2] + ['...'] + parts[-2:])
    return exp_id

compliance_long['exp_id_short'] = compliance_long['experiment_id'].apply(shorten_experiment_id)

# Pivot for heatmap: metrics x experiments
pivot_df = compliance_long.pivot_table(
    index='metric', columns='exp_id_short', values='within_threshold'
)

# Sort experiments by category mean compliance
mean_compliance = compliance_long.groupby('experiment_id')['within_threshold'].mean()
exp_order = mean_compliance.sort_values(ascending=False).index
pivot_df = pivot_df[compliance_long.set_index('experiment_id').loc[exp_order]['exp_id_short'].unique()]

# --- Create the heatmap ---
plt.figure(figsize=(20, 6))
sns.set_theme(style="whitegrid", font_scale=1.1)

# Convert True/False/NaN → numeric
heat_data = pivot_df.replace({True: 1, False: 0})

# Define color palette: green=1, red=0, gray=NaN
cmap = sns.color_palette(["#c62828", "#f5f5f5", "#2e7d32"])
sns.heatmap(
    heat_data,
    cmap=["#c62828", "#f5f5f5", "#2e7d32"],
    cbar=False,
    linewidths=0.5,
    linecolor='lightgray',
    square=True
)

# --- Annotate experiment-level compliance (mean %) above each column ---
col_means = heat_data.mean().round(2)
for i, (exp, val) in enumerate(col_means.items()):
    plt.text(i + 0.5, -0.3, f"{int(val * 100)}%", ha='center', va='bottom', fontsize=9, color='gray', fontweight='bold')

plt.title("Experiment Compliance Matrix\n(✓ within threshold, ✗ outside)", fontsize=14, fontweight='bold', pad=20)
plt.xlabel("Experiment ID", fontsize=12)
plt.ylabel("Metric", fontsize=12)

plt.xticks(rotation=90, ha='center', fontsize=9)
plt.yticks(fontsize=10)
plt.tight_layout()
plt.show()


## 9. Filtered Analysis: Experiments with Actual Degradation

Some experiments had chaos active but no actual degradation in latency or success rate. In these cases, the agent correctly left the problem location blank ([]). This section filters to only experiments with actual degradation for more meaningful location identification analysis.


In [ ]:
# Identify experiments with actual degradation
# An experiment has degradation if either latency or success rate degraded during chaos
has_latency_degradation = df['latency_degradation_pct'] > 1.0  # More than 1% latency increase
has_success_degradation = df['success_degradation_pct'] > 1.0  # More than 1% success rate decrease
has_any_degradation = has_latency_degradation | has_success_degradation

# Filter to only experiments with actual degradation
df_filtered = df[has_any_degradation].copy()

print(f"Total experiments: {len(df)}")
print(f"Experiments with actual degradation: {len(df_filtered)}")
print(f"Experiments excluded (no degradation): {len(df) - len(df_filtered)}")
print(f"\nExcluded experiments:")
excluded = df[~has_any_degradation][['experiment_id', 'category', 'latency_degradation_pct', 'success_degradation_pct', 'location_identified']]
print(excluded.to_string(index=False))

print(f"\n{'='*80}")
print("FILTERED DATASET STATISTICS")
print(f"{'='*80}")

# Location identification rate on filtered dataset
location_rate_filtered = df_filtered['location_identified'].mean() * 100
print(f"\nLocation Identification Rate (filtered): {location_rate_filtered:.1f}%")
print(f"Location Identification Rate (all experiments): {df['location_identified'].mean() * 100:.1f}%")

# Recovery success rate on filtered dataset
recovery_rate_filtered = df_filtered['recovery_success'].mean() * 100
print(f"\nRecovery Success Rate (filtered): {recovery_rate_filtered:.1f}%")
print(f"Recovery Success Rate (all experiments): {df['recovery_success'].mean() * 100:.1f}%")


In [ ]:
# Recovery vs Location Identification on filtered dataset
recovery_vs_location_filtered = (
    df_filtered.groupby('category')[['recovery_success', 'location_identified']]
      .mean()
      .rename(columns={'recovery_success': 'Recovery success', 'location_identified': 'Location identified'})
      .reset_index()
)

fig, ax = plt.subplots(figsize=(14, 6))
width = 0.35
positions = np.arange(len(recovery_vs_location_filtered))

ax.bar(positions - width/2, recovery_vs_location_filtered['Recovery success'] * 100, width, label='Recovery success', color='#1e88e5')
ax.bar(positions + width/2, recovery_vs_location_filtered['Location identified'] * 100, width, label='Location identified', color='#fdd835')

ax.set_xticks(positions)
ax.set_xticklabels(recovery_vs_location_filtered['category'], rotation=45, ha='right')
ax.set_ylabel('Rate (%)')
ax.set_title('Recovery vs. Location Identification (Experiments with Actual Degradation Only)')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


## 9. Aggregate Metrics Across All Experiments

Overall averages for degradation, recovery, and success-related metrics across the entire dataset.

In [ ]:
# Calculate comprehensive aggregate metrics
duration_mean = df['agent_duration_minutes'].mean()
duration_std = df['agent_duration_minutes'].std()
duration_min = df['agent_duration_minutes'].min()
duration_max = df['agent_duration_minutes'].max()

sleep_mean = df['agent_sleep_minutes'].mean()
sleep_std = df['agent_sleep_minutes'].std()
sleep_min = df['agent_sleep_minutes'].min()
sleep_max = df['agent_sleep_minutes'].max()

active_mean = df['agent_active_minutes'].mean()
active_std = df['agent_active_minutes'].std()

rounds_mean = df['agent_steps'].mean()
rounds_std = df['agent_steps'].std()
rounds_min = df['agent_steps'].min()
rounds_max = df['agent_steps'].max()

# Calculate average number of unique actions per experiment
def count_unique_actions(filepath):
    try:
        if filepath and filepath.exists():
            with open(filepath) as f:
                data = json.load(f)
            agent_trace = data.get('agent_trace', [])
            actions = [step.get('action') for step in agent_trace if step.get('action')]
            return len(set(actions)) if actions else 0
    except:
        pass
    return 0

# Add unique actions count to dataframe if not already present
if 'unique_actions' not in df.columns:
    # Create a mapping from filename to full path using available_files from cell 3
    file_map = {fp.name: fp for fp in available_files}
    df['unique_actions'] = df['file'].apply(lambda f: count_unique_actions(file_map.get(f)))

actions_mean = df['unique_actions'].mean()
actions_std = df['unique_actions'].std()
actions_min = df['unique_actions'].min()
actions_max = df['unique_actions'].max()

# Build comprehensive metrics table
overall_metrics = {
    'Metric': [
        'Latency Degradation During Chaos (%)',
        'Success-Rate Degradation During Chaos (%)',
        'Latency Recovery Index (%)',
        'Success-Rate Recovery Index (%)',
        'Composite Recovery Index (%)',
        'Experiments Meeting Both Thresholds (%)',
        'Experiments with Location Identified (%)',
        'Average Agent Duration (minutes)',             # Combined Mean ± Std Dev
        'Agent Duration Range (min-max)',
        'Average Sleep Time (minutes)',               # Combined Mean ± Std Dev
        'Sleep Time Range (min-max)',
        'Sleep Time as % of Duration',
        'Average Active Time (minutes)',              # Combined Mean ± Std Dev
        'Average Agent Rounds',                       # Combined Mean ± Std Dev
        'Agent Rounds Range (min-max)',
        'Average Unique Actions per Experiment',      # Combined Mean ± Std Dev
        'Unique Actions Range (min-max)',
    ],
    'Value': [
        f"{df['latency_degradation_pct'].mean():.2f}",
        f"{df['success_degradation_pct'].mean():.2f}",
        f"{df['latency_recovery_pct'].mean():.2f}",
        f"{df['success_recovery_pct'].mean():.2f}",
        f"{df['recovery_index_composite'].mean():.2f}",
        f"{df['recovery_success'].mean() * 100:.2f}",
        f"{df['location_identified'].mean() * 100:.2f}",
        f"{duration_mean:.2f} ± {duration_std:.2f}",  # Updated
        f"{duration_min:.2f} - {duration_max:.2f}",
        f"{sleep_mean:.2f} ± {sleep_std:.2f}",        # Updated
        f"{sleep_min:.2f} - {sleep_max:.2f}",
        f"{(sleep_mean / duration_mean * 100) if duration_mean > 0 else 0:.2f}",
        f"{active_mean:.2f} ± {active_std:.2f}",      # Updated
        f"{rounds_mean:.1f} ± {rounds_std:.1f}",      # Updated
        f"{int(rounds_min)} - {int(rounds_max)}",
        f"{actions_mean:.1f} ± {actions_std:.1f}",    # Updated
        f"{int(actions_min)} - {int(actions_max)}",
    ]
}

overall_df = pd.DataFrame(overall_metrics)
print("\n" + "="*80)
print("COMPREHENSIVE AGGREGATE METRICS ACROSS ALL EXPERIMENTS")
print("="*80)
print(overall_df.to_string(index=False))
print("="*80)

## 10. Comparison Across App Types

Compare key metrics across different application types to identify patterns and differences in agent performance.


In [ ]:
# Calculate key metrics per app type
app_comparison = df.groupby('app_type').agg({
    # Recovery metrics
    'latency_degradation_pct': ['mean', 'std'],
    'success_degradation_pct': ['mean', 'std'],
    'latency_recovery_pct': ['mean', 'std'],
    'success_recovery_pct': ['mean', 'std'],
    'recovery_index_composite': ['mean', 'std'],
    'recovery_success': 'mean',
    'location_identified': 'mean',
    # Agent effort metrics
    'agent_duration_minutes': ['mean', 'std', 'min', 'max'],
    'agent_sleep_minutes': ['mean', 'std'],
    'agent_active_minutes': ['mean', 'std'],
    'agent_steps': ['mean', 'std', 'min', 'max'],
    # Count experiments
    'experiment_id': 'count'
}).round(2)

# Flatten column names
app_comparison.columns = ['_'.join(col).strip() for col in app_comparison.columns.values]
app_comparison = app_comparison.reset_index()
app_comparison.rename(columns={'experiment_id_count': 'num_experiments'}, inplace=True)

# Create comparison table with key metrics
comparison_table = pd.DataFrame({
    'App Type': app_comparison['app_type'],
    'Num Experiments': app_comparison['num_experiments'].astype(int),
    'Recovery Success Rate (%)': (app_comparison['recovery_success_mean'] * 100).round(1),
    'Location Identified Rate (%)': (app_comparison['location_identified_mean'] * 100).round(1),
    'Avg Composite Recovery Index (%)': app_comparison['recovery_index_composite_mean'].round(1),
    'Avg Latency Recovery (%)': app_comparison['latency_recovery_pct_mean'].round(1),
    'Avg Success Rate Recovery (%)': app_comparison['success_recovery_pct_mean'].round(1),
    'Avg Duration (min)': app_comparison['agent_duration_minutes_mean'].round(2),
    'Duration Range (min-max)': (
        app_comparison['agent_duration_minutes_min'].round(1).astype(str) + ' - ' +
        app_comparison['agent_duration_minutes_max'].round(1).astype(str)
    ),
    'Avg Sleep Time (min)': app_comparison['agent_sleep_minutes_mean'].round(2),
    'Sleep % of Duration': (
        (app_comparison['agent_sleep_minutes_mean'] / 
         app_comparison['agent_duration_minutes_mean'] * 100).round(1).astype(str) + '%'
    ),
    'Avg Rounds': app_comparison['agent_steps_mean'].round(1),
    'Rounds Range (min-max)': (
        app_comparison['agent_steps_min'].astype(int).astype(str) + ' - ' +
        app_comparison['agent_steps_max'].astype(int).astype(str)
    ),
})

print("\n" + "="*100)
print("KEY METRICS COMPARISON BY APP TYPE")
print("="*100)
print(comparison_table.to_string(index=False))
print("="*100)

In [ ]:
# Create comparison visualizations
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

app_types = app_comparison['app_type'].values
x_pos = np.arange(len(app_types))
width = 0.6

# 1. Recovery Success Rate
axes[0, 0].bar(x_pos, app_comparison['recovery_success_mean'] * 100, width, 
               color='#2e7d32', alpha=0.8)
axes[0, 0].set_title('Recovery Success Rate', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Rate (%)', fontsize=10)
axes[0, 0].set_xticks(x_pos)
axes[0, 0].set_xticklabels(app_types, rotation=15, ha='right', fontsize=9)
axes[0, 0].grid(axis='y', alpha=0.3)
axes[0, 0].set_ylim([0, 100])

# 2. Location Identification Rate
axes[0, 1].bar(x_pos, app_comparison['location_identified_mean'] * 100, width,
               color='#f57c00', alpha=0.8)
axes[0, 1].set_title('Location Identification Rate', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Rate (%)', fontsize=10)
axes[0, 1].set_xticks(x_pos)
axes[0, 1].set_xticklabels(app_types, rotation=15, ha='right', fontsize=9)
axes[0, 1].grid(axis='y', alpha=0.3)
axes[0, 1].set_ylim([0, 100])

# 3. Composite Recovery Index
axes[0, 2].bar(x_pos, app_comparison['recovery_index_composite_mean'], width,
               color='#1e88e5', alpha=0.8)
axes[0, 2].set_title('Composite Recovery Index', fontsize=12, fontweight='bold')
axes[0, 2].set_ylabel('Index (%)', fontsize=10)
axes[0, 2].set_xticks(x_pos)
axes[0, 2].set_xticklabels(app_types, rotation=15, ha='right', fontsize=9)
axes[0, 2].grid(axis='y', alpha=0.3)

# 4. Average Duration
axes[1, 0].bar(x_pos, app_comparison['agent_duration_minutes_mean'], width,
               color='#3949ab', alpha=0.8)
axes[1, 0].set_title('Average Agent Duration', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Duration (minutes)', fontsize=10)
axes[1, 0].set_xticks(x_pos)
axes[1, 0].set_xticklabels(app_types, rotation=15, ha='right', fontsize=9)
axes[1, 0].grid(axis='y', alpha=0.3)

# 5. Average Rounds
axes[1, 1].bar(x_pos, app_comparison['agent_steps_mean'], width,
               color='#00897b', alpha=0.8)
axes[1, 1].set_title('Average Agent Rounds', fontsize=12, fontweight='bold')
axes[1, 1].set_ylabel('Rounds', fontsize=10)
axes[1, 1].set_xticks(x_pos)
axes[1, 1].set_xticklabels(app_types, rotation=15, ha='right', fontsize=9)
axes[1, 1].grid(axis='y', alpha=0.3)

# 6. Sleep Time as % of Duration
sleep_pct = (app_comparison['agent_sleep_minutes_mean'] / 
             app_comparison['agent_duration_minutes_mean'] * 100)
axes[1, 2].bar(x_pos, sleep_pct, width,
               color='#f57c00', alpha=0.8)
axes[1, 2].set_title('Sleep Time as % of Duration', fontsize=12, fontweight='bold')
axes[1, 2].set_ylabel('Percentage (%)', fontsize=10)
axes[1, 2].set_xticks(x_pos)
axes[1, 2].set_xticklabels(app_types, rotation=15, ha='right', fontsize=9)
axes[1, 2].grid(axis='y', alpha=0.3)

plt.suptitle('Key Metrics Comparison Across App Types', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

In [ ]:
# Detailed comparison table with degradation metrics
detailed_comparison = pd.DataFrame({
    'App Type': app_comparison['app_type'],
    'Num Experiments': app_comparison['num_experiments'].astype(int),
    'Avg Latency Degradation (%)': app_comparison['latency_degradation_pct_mean'].round(1),
    'Avg Success Rate Degradation (%)': app_comparison['success_degradation_pct_mean'].round(1),
    'Avg Latency Recovery (%)': app_comparison['latency_recovery_pct_mean'].round(1),
    'Avg Success Rate Recovery (%)': app_comparison['success_recovery_pct_mean'].round(1),
    'Composite Recovery Index (%)': app_comparison['recovery_index_composite_mean'].round(1),
    'Recovery Success Rate (%)': (app_comparison['recovery_success_mean'] * 100).round(1),
    'Location Identified Rate (%)': (app_comparison['location_identified_mean'] * 100).round(1),
    'Avg Duration (min)': app_comparison['agent_duration_minutes_mean'].round(2),
    'Duration Std Dev (±)': app_comparison['agent_duration_minutes_std'].round(2),
    'Avg Sleep (min)': app_comparison['agent_sleep_minutes_mean'].round(2),
    'Avg Active (min)': app_comparison['agent_active_minutes_mean'].round(2),
    'Avg Rounds': app_comparison['agent_steps_mean'].round(1),
    'Rounds Std Dev (±)': app_comparison['agent_steps_std'].round(1),
})

print("\n" + "="*120)
print("DETAILED METRICS COMPARISON BY APP TYPE")
print("="*120)
print(detailed_comparison.to_string(index=False))
print("="*120)

## 10. Export Processed Dataset

Save the processed per-experiment dataset for downstream analysis.

In [ ]:
# output_path = Path('processed_experiments.csv')
# df.to_csv(output_path, index=False)
# print(f'Saved processed dataset to {output_path.resolve()}')

### Key Takeaways

- **Degradation vs. recovery:** The summary table quantifies average chaos degradation relative to baseline and how much of that degradation the system regained.
- **Recovery vs. detection:** Comparing recovery success with location identification rates indicates whether the agent not only recovers the system but also diagnoses the fault.
- **Agent effort:** Runtime and step counts highlight which experiment types require the most effort from the agent.
- **Threshold compliance:** The scatter plot shows any experiments falling outside the latency or success-rate thresholds, while the within-20% comparison validates log integrity.
- **Global view:** The final bar chart aggregates the most important metrics across the entire dataset for a quick health check.